---
title: "术语提取"
date: "2025-09-20"
excerpt: "在从海量政策文件中用大模型提取术语时，冗余成分较多。本文介绍一种 Embedding + 聚类 + 大模型的术语归一化方法。"
tags: ["NLP", "术语提取", "jieba", "embedding"]
category: "技术"
language: "zh"
---

## 一、问题背景

### 1.1 问题描述

![和老师的聊天记录](/images/posts/image1.png)

最近遇到了一个技术问题，在从海量的政策文件中用大模型提取术语时，冗余成分比较多。

比如围绕 **"GPU"** 这个概念，AI 提取的术语可能包括：

- 高性能 GPU
- WebGPU
- GPU 服务器
- Nvidia H100 GPU
- 英伟达 A100 GPU
- GH200 超级 GPU
- Blackwell GPU
- H800 GPU

但实际上，在很多统计或指标计算场景中，**"GPU"这个上位概念就已经能够概括以上所有内容**。

> **那 A100 这种具体型号怎么办？**

具体型号其实也被单独提取和记录下来了；但在做整体统计、统一口径时，优先用 **"GPU"** 这个总称，避免同义变体导致的重复统计或口径不一致。

所以，我们的目标是在 **"不重不漏"** 的前提下，把术语表尽量简化、表达准确，为后续的指标测算打好基础。

![术语提取](/images/posts/image2.png)

## 二、方法尝试

### 2.1 传统分词：jieba

首先想到的就是直接用现成的 **jieba 分词库**。

但问题是，分词会把一些本来是完整的术语拆掉。例如图中的 **"指示学习"**，就很有可能被分成：

```
指示 / 学习
```

而在 NLP 或机器学习语境中，**"指示学习（Instruction Learning）"其实是一个完整术语**。  
因此，单纯依赖分词并不能很好解决术语识别的问题。

In [2]:
import jieba

# 示例：jieba 分词可能将完整术语拆分
text = "指示学习是机器学习的一个重要分支"
words = jieba.lcut(text)
print("分词结果:", words)
# 输出：['指示', '学习', '是', '机器', '学习', '的', '一个', '重要', '分支']
# 问题："指示学习" 被拆分成 "指示 / 学习"

分词结果：['指示', '学习', '是', '机器', '学习', '的', '一个', '重要', '分支']


### 2.2 正则表达式

另一个思路是使用 **正则表达式**。

但正则表达式有一个明显的问题：  
必须 **提前知道所有术语** 才能写出规则。

而我们的任务恰恰是 **从海量文本中发现术语**，术语本身是动态产生的，因此这种方法并不适用。

### 2.3 直接使用大模型

再有就是直接让 **大模型处理术语合并**。

理论上来说，"解决不了的问题上大模型"似乎是一个万能方案，但实际尝试后发现有几个问题：

1. 这些术语之间**语义非常相近**，很难写出稳定准确的提示词；
2. 大模型**长上下文比对能力有限**，如果要在一个长列表里逐个比较术语，复杂度接近 **O(n²)**；
3. 如果把任务拆成 **n 次调用**（每次判断一个词），虽然复杂度变成 **O(n)**，但数据量太大，**调用成本非常高**。

因此，单纯依赖大模型并不是一个现实的解决方案。

## 三、可行方案：Embedding + 聚类 + 大模型

### 3.1 整体思路

先把术语映射到向量空间，再根据语义相似度进行聚类。

**处理流程：**

```
术语抽取 → BERT Embedding → K-means 聚类 → LLM 选择上位词 → 统一术语表
```

### 3.2 第一步：Embedding 表示

使用预训练模型把术语转换成向量表示。

这里我们使用的是 `bert-base-chinese`，原因：
- 中文语义表达能力比较强
- 生态成熟
- 在短文本语义表示上效果不错

In [3]:
from transformers import AutoTokenizer, AutoModel
import torch

# 加载中文 BERT 模型
model_name = "bert-base-chinese"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

def get_embedding(text: str) -> torch.Tensor:
    """获取文本的 BERT embedding 表示"""
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
    # 使用 [CLS] token 的表示作为整个句子的嵌入
    return outputs.last_hidden_state[:, 0, :]

# 示例
terms = ["GPU", "高性能 GPU", "A100 GPU", "指示学习", "机器学习"]
embeddings = [get_embedding(term) for term in terms]
print(f"术语向量维度：{embeddings[0].shape}")

/opt/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11091.91it/s]
BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/archit

术语向量维度：torch.Size([1, 768])


### 3.3 第二步：语义聚类

有了向量之后，就可以计算 **语义相似度**，使用 **k-means 聚类** 把语义相近的术语自动分到同一个簇里。

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# 设置环境变量来抑制 tqdm 警告
import os
os.environ['TQDM_DISABLE'] = '1'

from sklearn.cluster import KMeans
import numpy as np

# 假设有以下术语列表
terms = [
    "GPU", "高性能 GPU", "GPU 服务器", "A100 GPU", "H100 GPU",
    "机器学习", "深度学习", "强化学习", "监督学习",
    "指示学习", "指令微调", "指令跟随",
    "Transformer", "注意力机制", "自注意力"
]

# 获取所有术语的 embedding
embedding_matrix = torch.vstack([get_embedding(term) for term in terms]).numpy()

# K-means 聚类
n_clusters = 4  # 假设我们知道有 4 个语义簇
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
labels = kmeans.fit_predict(embedding_matrix)

# 显示聚类结果
for i in range(n_clusters):
    cluster_terms = [terms[j] for j in range(len(terms)) if labels[j] == i]
    print(f"Cluster {i + 1}: {cluster_terms}")

Cluster 1: ['机器学习', '深度学习', '强化学习', '监督学习', '指示学习']
Cluster 2: ['GPU', '高性能 GPU', 'GPU 服务器', 'A100 GPU', 'H100 GPU', 'Transformer']
Cluster 3: ['指令微调', '指令跟随']
Cluster 4: ['注意力机制', '自注意力']


### 3.4 第三步：大模型选择上位词

聚类之后，每个 cluster 里面会同时存在：
- **上位词**（如：GPU）
- **下位词**（如：A100 GPU、H100 GPU）

从统计口径来看，我们需要一个 **代表性术语（canonical term）**。

此时让大模型在小规模、语义稳定的集合中做判断，准确率会明显提升，调用成本也能控制。

In [ ]:
# 示例：构造大模型提示词
def build_prompt(cluster_terms: list) -> str:
    """构建让大模型选择上位词的提示词"""
    terms_str = "\n".join([f"- {term}" for term in cluster_terms])
    prompt = f"""请从以下术语中选出最合适的上位概念（能够概括其他所有术语的总称）。

术语列表：
{terms_str}

要求：
1. 只输出一个术语，不要解释
2. 选择能够代表整个簇的上位词

选出的上位词："""
    return prompt

# 示例簇
cluster_a = ["GPU", "高性能 GPU", "GPU 服务器", "A100 GPU", "H100 GPU"]
print(build_prompt(cluster_a))

## 四、总结

### 4.1 最终方案

| 步骤 | 方法 | 作用 |
|------|------|------|
| 1. 术语抽取 | 大模型/规则 | 从文本中提取候选术语 |
| 2. Embedding | BERT | 将术语映射到向量空间 |
| 3. 聚类 | K-means | 语义相近的术语归为一类 |
| 4. 上位词选择 | 大模型 | 每个簇选出一个代表性术语 |

### 4.2 关键优势

1. **Embedding + 聚类** 将 O(n²) 的比对复杂度降为 O(n)
2. **聚类后的大模型调用** 只在小规模集合中做判断，成本可控
3. **语义驱动** 而非规则驱动，适应性强